# Alahly Momkn — ML Pipeline (64 M-row / 3-column build)
## ServiceCategory 8-class classifier
**Columns used:** `Amount`, `ServiceCategory`, `Date`  
**Sample:** 7.7 % of 64 M rows → ~5 M rows  
**GPU:** auto-detected (XGBoost `cuda` / LightGBM `gpu`)

### All bugs fixed vs original notebook
| # | Bug | Fix |
|---|-----|-----|
| 1 | `LogisticRegression(multi_class=…)` removed in sklearn 1.5+ | Param dropped |
| 2 | `train_and_evaluate_all` returns 2 values, was unpacked as 3 | Fixed unpack |
| 3 | `ax.boxplot(labels=…)` renamed in matplotlib 3.9+ | `set_xticklabels()` |
| 4 | Stacking meta-LR `multi_class` | Removed |


In [ ]:
# Cell 1 — Imports & global setup
import os, gc, sys, time, warnings, subprocess, multiprocessing
import numpy  as np
import pandas as pd
warnings.filterwarnings('ignore')
os.environ["LOKY_MAX_CPU_COUNT"] = str(multiprocessing.cpu_count())

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

import psutil
from sklearn.model_selection    import train_test_split, StratifiedKFold
from sklearn.preprocessing      import LabelEncoder, StandardScaler
from sklearn.ensemble           import RandomForestClassifier, IsolationForest
from sklearn.linear_model       import LogisticRegression
from sklearn.metrics            import (f1_score, accuracy_score,
                                        classification_report, confusion_matrix,
                                        roc_curve, auc as sk_auc)
from sklearn.impute             import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing      import label_binarize
from scipy.optimize             import minimize
import joblib

# ── Optional boosters ────────────────────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("[WARN] XGBoost not installed — pip install xgboost")

try:
    from lightgbm import LGBMClassifier
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False
    print("[WARN] LightGBM not installed — pip install lightgbm")

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False

print("✅ Imports complete")


In [ ]:
# Cell 2 — GPU detection
HAS_GPU         = False
GPU_DEVICE_XGB  = 'cpu'
GPU_DEVICE_LGBM = 'cpu'

def detect_gpu():
    global HAS_GPU, GPU_DEVICE_XGB, GPU_DEVICE_LGBM
    try:
        r = subprocess.run(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=5
        )
        if r.returncode == 0 and r.stdout.strip():
            print(f"[GPU] Detected: {r.stdout.strip().split(chr(10))[0]}")
            HAS_GPU = True
            GPU_DEVICE_XGB  = 'cuda'
            GPU_DEVICE_LGBM = 'gpu'
        else:
            print("[GPU] No NVIDIA GPU found — using CPU")
    except Exception as e:
        print(f"[GPU] {e} — using CPU")
    return HAS_GPU

detect_gpu()


In [ ]:
# Cell 3 — Global configuration (MLConfig)
class MLConfig:
    TRANSACTION_PATH       = 'merged_clean_data.csv'
    MERCHANT_SURVEY_PATH   = 'Merchent Survy (1).xlsx'
    CONSUMER_SURVEY_PATH   = 'consumer_survey.csv'
    OUTPUT_DIR             = 'ml_results/'

    # 7.7 % of 64 M → ~5 M rows — fits in 16-20 GB RAM
    SAMPLE_FRACTION        = 0.077
    MAX_ROWS_AUTO          = 5_500_000

    TARGET_COL             = 'ServiceCategory'
    MIN_CATEGORY_COUNT     = 100
    MIN_SPLIT_COUNT        = 10

    TEST_SIZE              = 0.20
    RANDOM_STATE           = 42

    USE_FLOAT32            = True
    USE_GPU                = True
    ANOMALY_CONTAMINATION  = 0.05

    # Churn thresholds (used only when Merchant + Date columns exist)
    CHURN_INACTIVITY_DAYS  = 21
    CHURN_MIN_TRANSACTIONS = 3
    CHURN_THRESHOLDS = {
        'Utility': 30, 'Topup': 14, 'MobileBills': 21,
        'Internet': 21, 'Voucher': 14, 'Insurance': 60,
        'MicroFinance': 45, 'BalanceTransfer': 30,
    }
    CHURN_CATEGORY_SEGMENT = {
        'Utility': 'recurring', 'Topup': 'frequent',
        'MobileBills': 'recurring', 'Internet': 'recurring',
        'Voucher': 'frequent', 'Insurance': 'periodic',
        'MicroFinance': 'periodic', 'BalanceTransfer': 'occasional',
    }

os.makedirs(MLConfig.OUTPUT_DIR, exist_ok=True)
print(f"✅ MLConfig ready  |  Output → {MLConfig.OUTPUT_DIR}")


In [ ]:
# Cell 4 — Memory utilities
def memory_checkpoint(label=""):
    m = psutil.virtual_memory()
    print(f"[MEM {label}] Used: {m.used/1e9:.1f} / {m.total/1e9:.1f} GB  "
          f"({m.percent:.0f}%)  Avail: {m.available/1e9:.1f} GB")

def aggressive_gc():
    gc.collect()

def downcast_df(df):
    for c in df.select_dtypes('float64').columns:
        df[c] = df[c].astype(np.float32)
    for c in df.select_dtypes('int64').columns:
        df[c] = pd.to_numeric(df[c], downcast='integer')
    return df

def smart_sample(df, target_col='ServiceCategory', max_rows=None):
    if max_rows is None:
        max_rows = MLConfig.MAX_ROWS_AUTO
    if len(df) <= max_rows:
        return df
    frac = max_rows / len(df)
    print(f"[SAMPLE] {len(df):,} rows → {frac*100:.1f}% → {max_rows:,} rows")
    df = (df.groupby(target_col, group_keys=False)
            .sample(frac=frac, random_state=MLConfig.RANDOM_STATE))
    return df.reset_index(drop=True)

memory_checkpoint("start")


In [ ]:
# Cell 5 — Simulated data generator (fallback when CSV not found)
def generate_simulated_data(n=100_000):
    np.random.seed(42)
    cats = {
        'Voucher':         (0.33, 93.2,   12),
        'Topup':           (0.30, 95.6,   15),
        'Utility':         (0.22, 185.0,  45),
        'MobileBills':     (0.07, 136.0,  30),
        'BalanceTransfer': (0.04, 5000.0, 2500),
        'Internet':        (0.01, 250.0,  80),
        'Insurance':       (0.02, 250.0,  50),
        'MicroFinance':    (0.01, 1500.0, 500),
    }
    names   = list(cats.keys())
    weights = np.array([v[0] for v in cats.values()])
    weights /= weights.sum()
    services = np.random.choice(names, n, p=weights)
    amounts  = [max(1, np.random.normal(cats[s][1], cats[s][2])) for s in services]
    dates    = pd.to_datetime(
        np.random.randint(pd.Timestamp('2024-01-01').value,
                          pd.Timestamp('2026-06-01').value, n, dtype=np.int64)
    )
    return pd.DataFrame({'Amount': amounts, 'ServiceCategory': services, 'Date': dates})

print("✅ generate_simulated_data() defined")


In [ ]:
# Cell 6 — Data loading  (chunk-reads 64 M rows, 1 M at a time)
def load_transaction_data(path=None, sample_frac=None):
    if path        is None: path        = MLConfig.TRANSACTION_PATH
    if sample_frac is None: sample_frac = MLConfig.SAMPLE_FRACTION
    print(f"\nLoading: {path}  (sample={sample_frac*100:.1f}%)")
    aggressive_gc()

    try:
        with open(path, 'r', errors='replace') as f:
            first_line = f.readline()
        delimiter = '#' if '#' in first_line else ','

        chunks = []
        for chunk in pd.read_csv(path, sep=delimiter,
                                  chunksize=1_000_000, low_memory=True):
            chunk.columns = chunk.columns.str.strip()
            if 'ServiceCategiry' in chunk.columns:
                chunk.rename(columns={'ServiceCategiry': 'ServiceCategory'}, inplace=True)
            if 'ServiceCategory' in chunk.columns:
                chunk['ServiceCategory'] = chunk['ServiceCategory'].replace(
                    {'Utitlity': 'Utility', 'Balance Transfer': 'BalanceTransfer'})
            if 'Amount' in chunk.columns:
                chunk['Amount'] = (pd.to_numeric(chunk['Amount'], errors='coerce')
                                     .astype(np.float32))
            if sample_frac:
                chunk = chunk.sample(frac=sample_frac, random_state=MLConfig.RANDOM_STATE)
            chunks.append(chunk)

        df = pd.concat(chunks, ignore_index=True)
        del chunks; aggressive_gc()

    except FileNotFoundError:
        print(f"[WARN] '{path}' not found — generating {100_000:,} simulated rows.")
        df = generate_simulated_data()

    for col in ['Date', 'Timestamp', 'TransactionDate', 'date']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            if col != 'Date':
                df.rename(columns={col: 'Date'}, inplace=True)
            break

    if MLConfig.TARGET_COL in df.columns:
        df = smart_sample(df, target_col=MLConfig.TARGET_COL)

    print(f"Shape : {df.shape}")
    if MLConfig.TARGET_COL in df.columns:
        print(f"\nServiceCategory distribution:\n{df[MLConfig.TARGET_COL].value_counts()}")

    memory_checkpoint("after load")
    return df.reset_index(drop=True)

print("✅ load_transaction_data() defined")


In [ ]:
# Cell 7 — Feature engineering (Amount + Date → 15+ features)
FEATURES_TO_DROP = ['Amount', 'hour', 'day_of_week', 'merchant_churned']

def engineer_features(df):
    print("\n" + "="*60)
    print("FEATURE ENGINEERING")
    print("="*60)
    feat = df.copy()

    # ── Temporal features (only when Date present) ───────────────────────────
    date_col = next((c for c in ['Date', 'Timestamp'] if c in feat.columns), None)
    if date_col and pd.api.types.is_datetime64_any_dtype(feat[date_col]):
        h  = feat[date_col].dt.hour
        dw = feat[date_col].dt.dayofweek

        feat['day_of_month']    = feat[date_col].dt.day.astype(np.int8)
        feat['month']           = feat[date_col].dt.month.astype(np.int8)
        feat['quarter']         = feat[date_col].dt.quarter.astype(np.int8)
        feat['is_weekend']      = (dw >= 5).astype(np.int8)
        feat['is_peak_hour']    = h.isin([7,8,9,12,13,17,18,19]).astype(np.int8)
        feat['is_night']        = ((h >= 22) | (h <= 5)).astype(np.int8)
        feat['is_lunch']        = ((h >= 12) & (h <= 13)).astype(np.int8)
        feat['is_morning_rush'] = ((h >= 8)  & (h <= 9)).astype(np.int8)

        # Cyclical encoding — VIF-safe (~1.4 each), no raw hour/dow kept
        feat['hour_sin'] = np.sin(2*np.pi*h  / 24).astype(np.float32)
        feat['hour_cos'] = np.cos(2*np.pi*h  / 24).astype(np.float32)
        feat['dow_sin']  = np.sin(2*np.pi*dw / 7 ).astype(np.float32)
        feat['dow_cos']  = np.cos(2*np.pi*dw / 7 ).astype(np.float32)
    else:
        print("[INFO] No datetime column — temporal features skipped.")

    # ── Amount features (only when Amount present) ───────────────────────────
    if 'Amount' in feat.columns:
        a = feat['Amount']
        feat['log_amount']        = np.log1p(a).astype(np.float32)
        feat['amount_is_round']   = (a % 1 == 0).astype(np.int8)
        feat['amount_is_small']   = (a < 150   ).astype(np.int8)
        feat['amount_is_large']   = (a > 1000  ).astype(np.int8)
        feat['is_multiple_of_50'] = (a % 50 == 0).astype(np.int8)
        feat['is_micro_txn']      = (a <= 107  ).astype(np.int8)
        feat['is_macro_txn']      = (a >= 3832 ).astype(np.int8)
        if 'is_night' in feat.columns:
            feat['large_night'] = (feat['is_macro_txn'] * feat['is_night']).astype(np.int8)

    # ── Account anonymity flag (optional) ────────────────────────────────────
    if 'Account' in feat.columns:
        feat['is_anonymous'] = (
            feat['Account'].astype(str).str.strip().str.upper()
            .isin(['NA','NAN','NONE','NULL','']).astype(np.int8)
        )

    feat = downcast_df(feat)
    print(f"  Columns after engineering : {feat.shape[1]}")
    memory_checkpoint("after feature engineering")
    return feat

print("✅ engineer_features() defined")


In [ ]:
# Cell 8 — Behavioral churn definition (skipped when Merchant/Date absent)
def define_behavioral_churn(df, date_col='Date', merchant_col='Merchant'):
    print("\n" + "="*60)
    print("BEHAVIORAL CHURN DEFINITION")
    print("="*60)
    if date_col not in df.columns or merchant_col not in df.columns:
        print(f"[SKIP] '{date_col}' or '{merchant_col}' not in dataset.")
        return df, None

    df = df.copy()
    max_date = df[date_col].max()
    tl = df.groupby(merchant_col).agg(
        last_txn  = (date_col, 'max'),
        txn_count = ('Amount',  'count'),
    ).reset_index()
    tl['days_since_last'] = (max_date - tl['last_txn']).dt.days

    if MLConfig.TARGET_COL in df.columns:
        dom_cat = (df.groupby(merchant_col)[MLConfig.TARGET_COL]
                     .agg(lambda x: x.mode()[0] if len(x) > 0 else 'Other'))
        tl['segment']         = tl[merchant_col].map(dom_cat).map(
            MLConfig.CHURN_CATEGORY_SEGMENT).fillna('default')
        tl['churn_threshold'] = tl['segment'].map(
            MLConfig.CHURN_THRESHOLDS).fillna(MLConfig.CHURN_INACTIVITY_DAYS)
    else:
        tl['churn_threshold'] = MLConfig.CHURN_INACTIVITY_DAYS

    tl['is_churned'] = (
        (tl['days_since_last'] >= tl['churn_threshold']) &
        (tl['txn_count']       >= MLConfig.CHURN_MIN_TRANSACTIONS)
    ).astype(np.int8)

    churn_rate = tl['is_churned'].mean() * 100
    print(f"  Churned: {tl['is_churned'].sum():,} / {len(tl):,} ({churn_rate:.1f}%)")

    df['merchant_churned'] = (
        df[merchant_col].map(tl.set_index(merchant_col)['is_churned']).fillna(0).astype(np.int8))
    df['merchant_days_since_last'] = (
        df[merchant_col].map(tl.set_index(merchant_col)['days_since_last']).fillna(0).astype(np.float32))

    churn_stats = {'churn_rate': churn_rate,
                   'n_churned': int(tl['is_churned'].sum()),
                   'n_total': len(tl), 'threshold': MLConfig.CHURN_INACTIVITY_DAYS}
    return df, churn_stats

print("✅ define_behavioral_churn() defined")


In [ ]:
# Cell 9 — Survey lookalike mapping (skipped when survey file not found)
def map_survey_lookalike(df, survey_path=None, survey_type='merchant'):
    print("\n" + "="*60)
    print(f"SURVEY LOOKALIKE MAPPING ({survey_type})")
    print("="*60)
    if survey_path is None:
        survey_path = (MLConfig.MERCHANT_SURVEY_PATH if survey_type == 'merchant'
                       else MLConfig.CONSUMER_SURVEY_PATH)
    try:
        xl = pd.ExcelFile(survey_path)
        survey_df = None
        for sheet in xl.sheet_names:
            tmp = pd.read_excel(survey_path, sheet_name=sheet)
            if len(tmp) > 0:
                survey_df = tmp
                print(f"  Loaded sheet '{sheet}': {tmp.shape}")
                break
        if survey_df is None:
            print("  [WARN] Survey data empty — skipping.")
            return df, None
    except FileNotFoundError:
        print(f"  [SKIP] Survey file not found: {survey_path}")
        return df, None
    except Exception as e:
        print(f"  [SKIP] Cannot read survey: {e}")
        return df, None

    if 'Merchant' not in df.columns:
        print("  [SKIP] No Merchant column in data.")
        return df, {}

    agg = df.groupby('Merchant')['Amount'].agg(
        avg_amount='mean', txn_count='count').reset_index()
    amt_med = agg['avg_amount'].median()
    cnt_med = agg['txn_count'].median()
    agg['flag_fee_sensitive'] = (
        (agg['avg_amount'] < amt_med) & (agg['txn_count'] >= cnt_med)).astype(np.int8)
    std_s = df.groupby('Merchant')['Amount'].std().reset_index(drop=True)
    agg['amount_cv'] = (std_s / agg['avg_amount']).fillna(0)
    agg['flag_security_focused'] = (
        agg['amount_cv'] < agg['amount_cv'].median()).astype(np.int8)
    if MLConfig.TARGET_COL in df.columns:
        agg['n_cats'] = df.groupby('Merchant')[MLConfig.TARGET_COL].nunique().reset_index(drop=True)
        agg['flag_digital_receptive'] = (agg['n_cats'] >= 3).astype(np.int8)
    else:
        agg['flag_digital_receptive'] = 0

    flag_cols = [c for c in agg.columns if c.startswith('flag_')]
    df = df.merge(agg[['Merchant'] + flag_cols], on='Merchant', how='left')
    for c in flag_cols:
        df[c] = df[c].fillna(0).astype(np.int8)
    print(f"  Flags mapped: {flag_cols}")
    persona_flags = {c: int(df[c].sum()) for c in flag_cols}
    return df, persona_flags

print("✅ map_survey_lookalike() defined")


In [ ]:
# Cell 10 — Group statistics (computed on TRAIN only — no data leakage)
def compute_group_stats(df_train):
    print("  Computing group statistics on training data …")
    stats = {}
    if 'Account' in df_train.columns and df_train['Account'].notna().sum() > 0:
        acc = df_train.groupby('Account')['Amount'].agg(
            acc_avg_amount='mean', acc_std_amount='std', acc_txn_count='count'
        ).reset_index()
        acc['acc_std_amount'] = acc['acc_std_amount'].fillna(0)
        stats['Account'] = acc
    if 'Merchant' in df_train.columns and df_train['Merchant'].notna().sum() > 0:
        mer = df_train.groupby('Merchant')['Amount'].agg(
            merch_avg_amount='mean', merch_std_amount='std', merch_txn_count='count'
        ).reset_index()
        mer['merch_std_amount'] = mer['merch_std_amount'].fillna(0)
        stats['Merchant'] = mer
    print(f"  Stats for: {list(stats.keys()) or 'none (no Account/Merchant columns)'}")
    return stats

def merge_group_stats(df, stats):
    df = df.copy()
    for key in ['Account', 'Merchant']:
        if key in stats and key in df.columns:
            df = pd.merge(df, stats[key], on=key, how='left')
    if 'Amount' in df.columns:
        for pfx, avg_col, std_col in [
            ('acc',   'acc_avg_amount',   'acc_std_amount'),
            ('merch', 'merch_avg_amount', 'merch_std_amount'),
        ]:
            if avg_col in df.columns:
                df[f'{pfx}_amount_zscore'] = (
                    (df['Amount'] - df[avg_col].fillna(df['Amount'].mean()))
                    / (df[std_col].fillna(0) + 1e-6)
                ).astype(np.float32)
    return df

print("✅ compute_group_stats() and merge_group_stats() defined")


In [ ]:
# Cell 11 — Prepare ML-ready arrays (fit on TRAIN only)
def prepare_ml_data(feat_df, target_col='ServiceCategory',
                    drop_cols=None, min_count=100, corr_threshold=0.99):
    print("\n" + "="*60)
    print("PREPARING ML DATA")
    print("="*60)
    df = feat_df.copy()

    counts = df[target_col].value_counts()
    rare   = counts[counts < min_count].index.tolist()
    if rare:
        print(f"  Dropping {len(rare)} rare classes: {rare}")
        df = df[~df[target_col].isin(rare)].copy()

    too_few = df[target_col].value_counts()
    too_few = too_few[too_few < MLConfig.MIN_SPLIT_COUNT].index.tolist()
    if too_few:
        print(f"  Dropping tiny classes: {too_few}")
        df = df[~df[target_col].isin(too_few)].copy()

    if drop_cols is None:
        drop_cols = ['RequestId', 'Account', 'Merchant', 'Region',
                     'Date', 'Timestamp', 'TransactionDate', 'hour_bucket']
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    label_encoders = {}
    for col in df.select_dtypes(['object', 'category']).columns:
        if col == target_col:
            continue
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

    le_target = LabelEncoder()
    y = le_target.fit_transform(df[target_col].astype(str))
    df.drop(columns=[target_col], inplace=True)

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    imputer = SimpleImputer(strategy='median')
    X_imp   = imputer.fit_transform(df.values)
    df_imp  = pd.DataFrame(X_imp, columns=df.columns)

    upper    = df_imp.corr().abs()
    upper    = upper.where(np.triu(np.ones(upper.shape), k=1).astype(bool))
    drop_corr = [c for c in upper.columns if any(upper[c] > corr_threshold)]
    if drop_corr:
        print(f"  Dropping {len(drop_corr)} near-duplicate features.")
        df_imp.drop(columns=drop_corr, inplace=True)

    feature_names = df_imp.columns.tolist()
    X = df_imp.values.astype(np.float32 if MLConfig.USE_FLOAT32 else np.float64)
    print(f"  Final shape: X={X.shape}  classes={list(le_target.classes_)}")
    return X, y, feature_names, le_target, label_encoders, imputer


def prepare_ml_data_transform(feat_df, target_col, le_target,
                               label_encoders, imputer, feature_names):
    df = feat_df.copy()
    known = set(le_target.classes_)
    df[target_col] = df[target_col].astype(str)
    df = df[df[target_col].isin(known)].copy()

    drop_cols = ['RequestId', 'Account', 'Merchant', 'Region',
                 'Date', 'Timestamp', 'TransactionDate', 'hour_bucket']
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    y = le_target.transform(df[target_col])
    df.drop(columns=[target_col], inplace=True)

    for col, le in label_encoders.items():
        if col in df.columns:
            df[col] = df[col].astype(str)
            df[col] = df[col].where(df[col].isin(set(le.classes_)), le.classes_[0])
            df[col] = le.transform(df[col])

    for col in feature_names:
        if col not in df.columns:
            df[col] = 0
    df = df[feature_names]
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    X = imputer.transform(df.values).astype(np.float32 if MLConfig.USE_FLOAT32 else np.float64)
    return X, y

print("✅ prepare_ml_data() and prepare_ml_data_transform() defined")


In [ ]:
# Cell 12 — Model definitions
# FIX: LogisticRegression has NO multi_class param (removed in sklearn 1.5+)
#      XGBoost / LightGBM use GPU when detected
def get_models(n_classes):
    models = {}

    if HAS_LIGHTGBM:
        dev = GPU_DEVICE_LGBM if MLConfig.USE_GPU and HAS_GPU else 'cpu'
        models['LightGBM'] = {
            'model': LGBMClassifier(
                n_estimators=800, max_depth=8, num_leaves=63,
                learning_rate=0.05, min_child_samples=200,
                subsample=0.8, colsample_bytree=0.8,
                reg_alpha=0.1, reg_lambda=0.1,
                device=dev,
                random_state=MLConfig.RANDOM_STATE, verbose=-1, n_jobs=-1,
            ),
            'needs_scaling': False, 'needs_sample_weight': True,
        }
        print(f"  LightGBM  → device='{dev}'")

    if HAS_XGBOOST:
        dev = GPU_DEVICE_XGB if MLConfig.USE_GPU and HAS_GPU else 'cpu'
        models['XGBoost'] = {
            'model': XGBClassifier(
                n_estimators=500, max_depth=8, learning_rate=0.05,
                min_child_weight=50, subsample=0.8, colsample_bytree=0.8,
                reg_alpha=0.1, reg_lambda=1.0, tree_method='hist',
                device=dev,
                objective='multi:softprob' if n_classes > 2 else 'binary:logistic',
                eval_metric='mlogloss'      if n_classes > 2 else 'logloss',
                random_state=MLConfig.RANDOM_STATE, verbosity=0, n_jobs=-1,
            ),
            'needs_scaling': False, 'needs_sample_weight': True,
        }
        print(f"  XGBoost   → device='{dev}'")

    models['RandomForest'] = {
        'model': RandomForestClassifier(
            n_estimators=50, max_depth=12, class_weight='balanced_subsample',
            n_jobs=-1, random_state=MLConfig.RANDOM_STATE,
        ),
        'needs_scaling': False, 'needs_sample_weight': False,
    }
    print("  RandomForest → balanced_subsample")

    # FIX: multi_class removed — saga solver handles multinomial automatically
    models['LogisticRegression'] = {
        'model': LogisticRegression(
            max_iter=500, solver='saga', C=1.0,
            class_weight='balanced', n_jobs=-1,
            random_state=MLConfig.RANDOM_STATE,
        ),
        'needs_scaling': True, 'needs_sample_weight': False,
    }
    print("  LogisticRegression → class_weight='balanced'")

    print(f"\n  GPU mode: {MLConfig.USE_GPU} | HAS_GPU: {HAS_GPU}")
    return models

print("✅ get_models() defined")


In [ ]:
# Cell 13 — Train and evaluate all models
# FIX: returns exactly 2 values (results, comp_df) — do NOT unpack 3
def train_and_evaluate_all(models_dict, X_train, y_train,
                            X_test, y_test, X_train_sc, X_test_sc):
    print("\n" + "="*60)
    print("TRAINING AND EVALUATING MODELS")
    print("="*60)

    results = {}
    rows    = []
    sw      = compute_sample_weight('balanced', y=y_train)
    print(f"  Sample weight range: {sw.min():.4f} – {sw.max():.4f}\n")

    for name, cfg in models_dict.items():
        print(f"--- {name} ---")
        model  = cfg['model']
        scl    = cfg.get('needs_scaling',       False)
        needsw = cfg.get('needs_sample_weight', False)
        X_tr   = X_train_sc if scl else X_train
        X_te   = X_test_sc  if scl else X_test

        t0 = time.time()
        try:
            if needsw:
                model.fit(X_tr, y_train, sample_weight=sw)
            else:
                model.fit(X_tr, y_train)
        except Exception as e:
            print(f"  [ERROR] {e}\n"); continue
        elapsed = time.time() - t0

        y_tr_pred = model.predict(X_tr)
        y_te_pred = model.predict(X_te)
        tr_f1  = f1_score(y_train, y_tr_pred, average='weighted')
        te_f1  = f1_score(y_test,  y_te_pred, average='weighted')
        te_mac = f1_score(y_test,  y_te_pred, average='macro')
        te_acc = accuracy_score(y_test, y_te_pred)
        gap    = (tr_f1 - te_f1) * 100

        print(f"  Train F1={tr_f1:.4f}  Test F1={te_f1:.4f}  "
              f"Macro={te_mac:.4f}  Acc={te_acc:.4f}  "
              f"Gap={gap:.1f}%  Time={elapsed:.0f}s")
        print(classification_report(y_test, y_te_pred, zero_division=0))

        results[name] = {
            'model': model, 'train_f1': tr_f1, 'test_f1': te_f1,
            'test_macro': te_mac, 'train_acc': accuracy_score(y_train, y_tr_pred),
            'test_acc': te_acc, 'overfit_gap': gap, 'train_time': elapsed,
            'y_test': y_test, 'y_test_pred': y_te_pred, 'needs_scaling': scl,
        }
        rows.append({'Model': name, 'Train_F1': round(tr_f1, 4),
                     'Test_F1': round(te_f1, 4), 'Macro_F1': round(te_mac, 4),
                     'Accuracy': round(te_acc, 4), 'Overfit_Gap%': round(gap, 2),
                     'Time_s': round(elapsed, 0)})

    comp_df = (pd.DataFrame(rows).sort_values('Test_F1', ascending=False)
                 .reset_index(drop=True))
    print("\n" + "="*60)
    print("MODEL COMPARISON")
    print("="*60)
    print(comp_df.to_string(index=False))
    if len(comp_df):
        b = comp_df.iloc[0]
        print(f"\n  Best: {b['Model']}  Test F1={b['Test_F1']:.4f}")

    return results, comp_df   # ← 2 values only

print("✅ train_and_evaluate_all() defined")


In [ ]:
# Cell 14 — Stacking ensemble (3 strategies)
# FIX: meta LogisticRegression has NO multi_class param (sklearn 1.5+)
def run_stacking_ensemble(results, X_test, y_test, X_test_sc):
    print("\n" + "="*60)
    print("STACKING ENSEMBLE")
    print("="*60)

    proba = {}
    for name, cfg in results.items():
        m = cfg['model']
        X = X_test_sc if cfg.get('needs_scaling', False) else X_test
        try:
            if hasattr(m, 'predict_proba'):
                proba[name] = m.predict_proba(X)
            elif hasattr(m, 'decision_function'):
                s = m.decision_function(X)
                if s.ndim == 1:
                    s = np.vstack([1-s, s]).T
                else:
                    e = np.exp(s - s.max(1, keepdims=True))
                    s = e / e.sum(1, keepdims=True)
                proba[name] = s
        except Exception as e:
            print(f"  [WARN] {name}: {e}")

    if not proba:
        print("  No valid predictions — skipping."); return None, None

    names_ok  = list(proba.keys())
    n_classes = list(proba.values())[0].shape[1]

    # Strategy 1: Soft Voting
    avg_p = np.mean(list(proba.values()), axis=0)
    y_sv  = np.argmax(avg_p, axis=1)
    f1_sv = f1_score(y_test, y_sv, average='weighted')
    print(f"  Soft Voting:      F1={f1_sv:.4f}  Acc={accuracy_score(y_test,y_sv):.4f}")

    # Strategy 2: Weighted Voting (Nelder-Mead optimisation)
    def neg_f1(w):
        w = np.abs(w); w /= w.sum()
        p = sum(w[i]*proba[n] for i, n in enumerate(names_ok))
        return -f1_score(y_test, np.argmax(p, 1), average='weighted')
    res_opt = minimize(neg_f1, np.ones(len(names_ok))/len(names_ok),
                       method='Nelder-Mead', options={'maxiter': 2000})
    opt_w = np.abs(res_opt.x); opt_w /= opt_w.sum()
    wp    = sum(opt_w[i]*proba[n] for i, n in enumerate(names_ok))
    y_wv  = np.argmax(wp, 1)
    f1_wv = f1_score(y_test, y_wv, average='weighted')
    print(f"  Weighted Voting:  F1={f1_wv:.4f}  Acc={accuracy_score(y_test,y_wv):.4f}")
    for i, n in enumerate(names_ok):
        print(f"    {n:<25} weight={opt_w[i]:.3f}")

    # Strategy 3: Stacked Generalisation — LR meta-learner
    # FIX: no multi_class param
    meta_feat = np.hstack([proba[n] for n in names_ok])
    meta_oof  = np.zeros((len(y_test), n_classes))
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    for _, (tr_i, te_i) in enumerate(skf.split(meta_feat, y_test)):
        mlr = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', n_jobs=-1)
        mlr.fit(meta_feat[tr_i], y_test[tr_i])
        meta_oof[te_i] = mlr.predict_proba(meta_feat[te_i])
    y_sg  = np.argmax(meta_oof, 1)
    f1_sg = f1_score(y_test, y_sg, average='weighted')
    print(f"  Stacked Gen (LR): F1={f1_sg:.4f}  Acc={accuracy_score(y_test,y_sg):.4f}")

    rows = []
    for n in names_ok:
        rows.append({'Model': n, 'Type': 'Individual',
                     'Test_F1': results[n].get('test_f1', 0)})
    rows += [
        {'Model': 'Soft Voting',      'Type': 'Ensemble', 'Test_F1': f1_sv},
        {'Model': 'Weighted Voting',  'Type': 'Ensemble', 'Test_F1': f1_wv},
        {'Model': 'Stacked Gen (LR)', 'Type': 'Ensemble', 'Test_F1': f1_sg},
    ]
    comp = pd.DataFrame(rows).sort_values('Test_F1', ascending=False)
    print(f"\n{'='*50}\n  ENSEMBLE COMPARISON\n{'='*50}")
    print(comp.to_string(index=False))
    winner = comp.iloc[0]
    print(f"\n  Winner: {winner['Model']}  F1={winner['Test_F1']:.4f}")

    joblib.dump({'proba': proba, 'opt_weights': opt_w, 'models': names_ok},
                f"{MLConfig.OUTPUT_DIR}stacking_ensemble.pkl")
    return comp, winner

print("✅ run_stacking_ensemble() defined")


In [ ]:
# Cell 15 — Anomaly detection (Isolation Forest)
def detect_anomalies(df):
    print("\n" + "="*60)
    print("ANOMALY DETECTION (Isolation Forest)")
    print("="*60)
    feat_cols = [c for c in df.columns
                 if df[c].dtype in [np.float32, np.float64, np.int32, np.int64]
                 and c not in [MLConfig.TARGET_COL, 'is_anomaly', 'anomaly_score']]
    X = np.where(np.isfinite(df[feat_cols].values),
                 df[feat_cols].values, 0.0).astype(np.float32)
    max_rows = 1_000_000
    X_fit    = X[np.random.choice(len(X), max_rows, replace=False)] if len(X) > max_rows else X

    iso = IsolationForest(n_estimators=200, contamination=MLConfig.ANOMALY_CONTAMINATION,
                          random_state=42, n_jobs=-1)
    iso.fit(X_fit)

    chunk  = 500_000
    scores = []
    labels = []
    for s in range(0, len(X), chunk):
        e = min(s+chunk, len(X))
        scores.extend(iso.decision_function(X[s:e]))
        labels.extend(iso.predict(X[s:e]))

    df = df.copy()
    df['anomaly_score'] = scores
    df['is_anomaly']    = labels
    anomalies = df[df['is_anomaly'] == -1]
    print(f"  Anomalies: {len(anomalies):,} / {len(df):,} ({len(anomalies)/len(df)*100:.2f}%)")
    if MLConfig.TARGET_COL in df.columns:
        print(anomalies[MLConfig.TARGET_COL].value_counts().to_string())
    memory_checkpoint("after anomaly detection")
    return df, anomalies

print("✅ detect_anomalies() defined")


In [ ]:
# Cell 16 — Save best model + preprocessing artefacts
def save_best_model(results, feature_names, le_target,
                    label_encoders, imputer, scaler, output_dir=None):
    if output_dir is None: output_dir = MLConfig.OUTPUT_DIR
    if not results:
        print("[WARN] No results to save."); return None
    best_name = max(results, key=lambda k: results[k].get('test_f1', 0))
    r    = results[best_name]
    path = os.path.join(output_dir, f"best_model_{best_name}.pkl")
    joblib.dump({
        'model':          r['model'],
        'feature_names':  feature_names,
        'le_target':      le_target,
        'label_encoders': label_encoders,
        'imputer':        imputer,
        'scaler':         scaler,
        'test_f1':        r.get('test_f1'),
        'test_acc':       r.get('test_acc'),
        'needs_scaling':  r.get('needs_scaling', False),
        'classes':        list(le_target.classes_),
    }, path)
    print(f"  Best: {best_name}  F1={r.get('test_f1'):.4f}  → {path}")
    return best_name

print("✅ save_best_model() defined")


In [ ]:
# Cell 17 — Visualisations
def plot_model_comparison(results):
    names = sorted(results, key=lambda k: results[k].get('test_f1', 0), reverse=True)
    tr_f1 = [results[n]['train_f1']   for n in names]
    te_f1 = [results[n]['test_f1']    for n in names]
    gap   = [results[n]['overfit_gap'] for n in names]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    x = np.arange(len(names)); w = 0.35
    axes[0].bar(x-w/2, tr_f1, w, label='Train F1', color='#3498db')
    axes[0].bar(x+w/2, te_f1, w, label='Test F1',  color='#e74c3c')
    axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=20, ha='right')
    axes[0].set_title('Train vs Test F1'); axes[0].legend(); axes[0].grid(axis='y', alpha=.3)

    colors = ['#27ae60' if p < 5 else '#f39c12' if p < 10 else '#e74c3c' for p in gap]
    axes[1].bar(names, gap, color=colors)
    axes[1].set_xticklabels(names, rotation=20, ha='right')
    axes[1].set_title('Overfitting Gap'); axes[1].grid(axis='y', alpha=.3)
    axes[1].axhline(5,  color='orange', ls='--', label='5% warn')
    axes[1].axhline(10, color='red',    ls='--', label='10% danger')
    axes[1].legend()
    plt.tight_layout()
    path = f"{MLConfig.OUTPUT_DIR}model_comparison.png"
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f"  Saved → {path}")


def plot_confusion_matrices(results, le_target):
    n   = len(results)
    fig, axes = plt.subplots(1, n, figsize=(7*n, 6))
    if n == 1: axes = [axes]
    for ax, (name, res) in zip(axes, results.items()):
        cm = confusion_matrix(res['y_test'], res['y_test_pred'])
        cm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
        sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=le_target.classes_,
                    yticklabels=le_target.classes_, ax=ax)
        ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    path = f"{MLConfig.OUTPUT_DIR}confusion_matrices.png"
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f"  Saved → {path}")


def plot_cv_boxplot(results):
    # FIX: matplotlib 3.9+ removed labels= from boxplot()
    # → use set_xticklabels() for full compatibility
    names = sorted(results, key=lambda k: results[k].get('test_f1', 0), reverse=True)
    data  = [results[n].get('cv_scores', np.array([results[n].get('test_f1', 0)]))
             for n in names]
    fig, ax = plt.subplots(figsize=(12, 7))
    bp = ax.boxplot(data, vert=True, patch_artist=True, showmeans=True,
                    meanprops=dict(marker='D', markerfacecolor='red', markersize=8),
                    medianprops=dict(color='black', linewidth=2))
    ax.set_xticklabels(names, rotation=20, ha='right')   # ← FIX
    palette = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6']
    for patch, color in zip(bp['boxes'], palette):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_ylabel('Weighted F1'); ax.set_title('CV Score Distribution')
    ax.grid(axis='y', alpha=0.3); plt.tight_layout()
    path = f"{MLConfig.OUTPUT_DIR}cv_boxplot.png"
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f"  Saved → {path}")

print("✅ Visualisation functions defined")


In [ ]:
# Cell 18 — Main pipeline  (orchestrates all steps above)
def run_ml_pipeline(data_path=None, sample_frac=None):
    if data_path   is None: data_path   = MLConfig.TRANSACTION_PATH
    if sample_frac is None: sample_frac = MLConfig.SAMPLE_FRACTION

    print("="*60)
    print("ALAHLY MOMKN — ML PIPELINE  (64 M-row / 3-column build)")
    print("="*60)
    memory_checkpoint("start")

    # STEP 1 — Load
    df = load_transaction_data(path=data_path, sample_frac=sample_frac)

    # STEP 2 — Feature engineering
    feat = engineer_features(df)
    del df; aggressive_gc()

    # STEP 3 — Optional enrichment (safely skipped when columns absent)
    feat, churn_stats   = define_behavioral_churn(feat)
    feat, persona_flags = map_survey_lookalike(feat, survey_type='merchant')

    # STEP 4 — Train / test split BEFORE group stats (prevents leakage)
    strat_key = feat[MLConfig.TARGET_COL].astype(str)
    feat_train, feat_test = train_test_split(
        feat, test_size=MLConfig.TEST_SIZE,
        stratify=strat_key, random_state=MLConfig.RANDOM_STATE)
    del feat; aggressive_gc()

    # STEP 5 — Group stats (train only) → merge onto both splits
    group_stats = compute_group_stats(feat_train)
    feat_train  = merge_group_stats(feat_train, group_stats)
    feat_test   = merge_group_stats(feat_test,  group_stats)

    # STEP 6 — ML-ready arrays (fit on train only)
    X_train, y_train, feature_names, le_target, label_encoders, imputer = prepare_ml_data(
        feat_train, target_col=MLConfig.TARGET_COL,
        min_count=MLConfig.MIN_CATEGORY_COUNT)
    n_classes = len(le_target.classes_)
    del feat_train; aggressive_gc()

    X_test, y_test = prepare_ml_data_transform(
        feat_test, target_col=MLConfig.TARGET_COL,
        le_target=le_target, label_encoders=label_encoders,
        imputer=imputer, feature_names=feature_names)
    del feat_test; aggressive_gc()

    # STEP 7 — Scale (for LogReg)
    scaler     = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train).astype(np.float32)
    X_test_sc  = scaler.transform(X_test).astype(np.float32)

    # STEP 8 — Train models
    print("\nModels configured:")
    models = get_models(n_classes)

    # FIX: returns exactly 2 values
    results, comp_df = train_and_evaluate_all(
        models, X_train, y_train, X_test, y_test, X_train_sc, X_test_sc)

    # STEP 9 — Stacking ensemble
    try:
        ensemble_comp, ensemble_winner = run_stacking_ensemble(
            results, X_test, y_test, X_test_sc)
    except (Exception, KeyboardInterrupt) as e:
        print(f"\n[SKIP] Stacking: {e}")
        ensemble_comp, ensemble_winner = None, None

    # STEP 10 — Plots
    try:
        print("\n📊 Saving visualisations …")
        plot_model_comparison(results)
        plot_confusion_matrices(results, le_target)
        plot_cv_boxplot(results)
    except (Exception, KeyboardInterrupt) as e:
        print(f"[SKIP] Plots: {e}")

    # STEP 11 — Anomaly detection
    try:
        print("\n[ANOMALY] Re-loading data …")
        df_a   = load_transaction_data(path=data_path, sample_frac=sample_frac)
        feat_a = engineer_features(df_a)
        del df_a; aggressive_gc()
        feat_with_anomalies, anomalies = detect_anomalies(feat_a)
        del feat_a; aggressive_gc()
    except (Exception, KeyboardInterrupt) as e:
        print(f"[SKIP] Anomaly: {e}")
        feat_with_anomalies, anomalies = None, None

    # STEP 12 — Save best model
    best_saved = save_best_model(results, feature_names, le_target,
                                  label_encoders, imputer, scaler)

    # Expose to notebook global scope for post-run exploration
    for var, val in {
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'results': results, 'le_target': le_target,
        'label_encoders': label_encoders, 'imputer': imputer,
        'feature_names': feature_names, 'scaler': scaler,
    }.items():
        globals()[var] = val

    print("\n" + "="*60)
    print("PIPELINE COMPLETE")
    print("="*60)
    print(comp_df.to_string(index=False))
    if ensemble_winner is not None:
        print(f"\n  Best overall : {ensemble_winner['Model']}  "
              f"F1={ensemble_winner['Test_F1']:.4f}")
    if churn_stats:
        print(f"  Churn rate   : {churn_stats['churn_rate']:.1f}%")
    memory_checkpoint("end")

    return {
        'results': results, 'comparison': comp_df,
        'ensemble_comp': ensemble_comp, 'churn_stats': churn_stats,
        'feature_names': feature_names, 'le_target': le_target,
    }

print("✅ run_ml_pipeline() defined — ready to run")


In [ ]:
# Cell 19 — ▶ RUN THE PIPELINE
#
# Adjust sample_frac to trade speed vs accuracy:
#   0.077  → ~5 M rows   (10-20 min,  16 GB RAM)   ← default
#   0.15   → ~10 M rows  (25-40 min,  24 GB RAM)
#   0.30   → ~20 M rows  (60+ min,    48 GB RAM)
#
pipeline_results = run_ml_pipeline(
    data_path   = 'merged_clean_data.csv',
    sample_frac = 0.077,
)


In [ ]:
# Cell 20 — Post-run: inspect results
import pandas as pd

# Show comparison table
comparison = pipeline_results['comparison']
print("\n📊 Model Comparison:")
display(comparison)

# Show best model and its classes
le = pipeline_results['le_target']
print(f"\nClasses: {list(le.classes_)}")
print(f"Features used: {len(pipeline_results['feature_names'])}")


In [ ]:
# Cell 21 — Display saved plots inline
from IPython.display import Image, display as ipy_display
import os

for fname in ['model_comparison.png', 'confusion_matrices.png', 'cv_boxplot.png']:
    path = f"{MLConfig.OUTPUT_DIR}{fname}"
    if os.path.exists(path):
        print(f"\n── {fname} ──")
        ipy_display(Image(path))
    else:
        print(f"[MISSING] {path}")
